# Train lại probe EK100 trên token đầy đủ đã trích (`ek100-tok-*`)

Notebook dùng 10 dataset token (train p0–p5, val-sub, cả 2 giao thức) để:
1. **Dựng chỉ mục chung** theo `narration_id`, vì `clip_id` đánh số lại từ 0 trong từng dataset. Notebook kiểm tra bảng lớp trùng với
   probe Meta, bỏ segment lặp, kiểm tra không có segment val nào trong train, và giữ lại khoảng 10% người tham gia của train để chọn epoch.
2. **Kiểm tra:** giải nén token int8, chạy lại predictor và probe Meta trên vài clip, so với logits đã lưu; đo tốc độ đọc đĩa. Nếu không
   khớp, notebook dừng.
3. **Train**, cho từng giao thức:
   - `meta`: probe Meta, không train (mốc so sánh);
   - `heads:<loss>`: chỉ train lại 3 lớp cuối, trên đầu ra pooler của Meta (tính một lần, nhanh);
   - `full:<loss>`: train lại **cả pooler và 3 lớp cuối**, khởi tạo từ probe Meta, đọc token trực tiếp từ đĩa (chậm).

   Loss: `bsm` (Balanced Softmax), `focal` (loss của Meta), `ce`, `la`; thêm `+compose` để action được ghép từ verb và noun.

Epoch được chọn trên người tham gia giữ lại của **train**. Val ở đây chỉ là **tập con** (khoảng 31% val), nên kết quả là so sánh tương
đối giữa các recipe với probe Meta trên **cùng các clip**, chưa phải con số để so với 32.7.

**Cài đặt:** GPU T4 x2 (dùng 1 GPU), Internet On. *Add Input* cả 10 dataset `ek100-tok-*` (để Public nếu thuộc tài khoản khác). Muốn chạy
tiếp sau khi bị ngắt: *Add Input* thêm output của lần trước (thư mục `tok`). Có thể chia 2 giao thức cho 2 tài khoản chạy song song
bằng `ANCHORS`.

In [ ]:
# ---- settings ----
ANCHORS = ["action_start", "official"]       # or one per account, e.g. ["official"]
RECIPES = "meta heads:bsm+compose heads:focal heads:la+compose full:bsm+compose"
FULL_EPOCHS = 3
MAX_HOURS = 11.0
# ------------------
import time
NOTEBOOK_START = time.time()


In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!git -C $R log --oneline -1
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
ANN = "/kaggle/working/ek100-ann"
!mkdir -p /tmp/ckpt && wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
!ls -lh /tmp/ckpt; nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Resume from an earlier run of this notebook (folder tok/ in an attached output), then build / reuse the index
import glob, shutil
TOK = "/kaggle/working/tok"
prev = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/tok/index.csv", recursive=True)]
if prev:
    shutil.copytree(prev[0], TOK, dirs_exist_ok=True); print("resumed from", prev[0])
if not os.path.exists(f"{TOK}/index.csv"):
    !python $R/scripts/tok_probe.py index --root /kaggle/input --train_csv $ANN/EPIC_100_train.csv \
        --val_csv $ANN/EPIC_100_validation.csv --out_dir $TOK
else:
    print(open(f"{TOK}/index_report.json").read()[:3000])


In [ ]:
# Dequantisation + stored logits reproduced? + disk read speed. Stops here on a mismatch.
!python $R/scripts/tok_probe.py verify --index $TOK/index.csv --vitl_ckpt /tmp/ckpt/vitl.pt \
    --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt --device cuda:0
import json
v = json.load(open(f"{TOK}/verify.json"))
assert v["ok"], "stored logits are not reproduced: send the output of this cell"
print("dequant:", v["dequant"], "| read speed (MB/s):", {k: x["mb_per_s"] for k, x in v["read_speed"].items()})


In [ ]:
# One anchor after the other; each run stops cleanly at the time budget and keeps what is done (results.json,
# frozen_pass.npz, checkpoints in tok/<anchor>/). Run again with this output attached to continue.
for anchor in ANCHORS:
    left = MAX_HOURS - (time.time() - NOTEBOOK_START) / 3600
    if left < 0.3:
        print("time budget reached before", anchor); break
    print(f"\n==================== {anchor} ({left:.1f} h left) ====================", flush=True)
    !python $R/scripts/tok_probe.py train --index $TOK/index.csv --anchor $anchor --recipes $RECIPES \
        --train_csv $ANN/EPIC_100_train.csv --val_csv $ANN/EPIC_100_validation.csv \
        --vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt --out_dir $TOK \
        --full_epochs $FULL_EPOCHS --max_hours {left - 0.2} --device cuda:0
!du -sh $TOK


Gửi lại output của cell index, cell verify và các bảng cuối của cell train (mỗi giao thức một bảng).